# PostgreSQL vector search demo

Ingestion and schema setup are handled by `uv run ingest-pgvector`. This notebook only demonstrates the reusable PostgreSQL search and RAG modules.

In [7]:
import psycopg
from openai import OpenAI

from vector_search.config import DATABASE_URL, get_ollama_api_key
from vector_search.embeddings import create_embedder
from vector_search.pgvector_store import PgVectorStore
from vector_search.rag_base import RAGPgVector

embedder = create_embedder()
ollama_client = OpenAI(
    base_url="https://ollama.com/v1",
    api_key=get_ollama_api_key(),
)
conn = psycopg.connect(DATABASE_URL)
store = PgVectorStore(conn)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
query = "I just discovered the course. Can I still join it?"
query_vector = embedder.encode(query)
results = store.search(
    query_vector=query_vector,
    course="llm-zoomcamp",
    num_results=5,
)
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'similarity': 0.8365048419387595},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.',
  'similarity': 0.524265035263748},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions'

In [12]:
assistant = RAGPgVector(
    embedder=embedder,
    store=store,
    llm_client=ollama_client,
)

assistant.rag("I just discovered the course. Can I still join it?")

'Yes, you can still join. However, if you want to receive a certificate, you must submit your project while submissions are still being accepted.'

In [13]:
conn.close()